# 뉴스 데이터 전처리 및 카테고리별 임베딩

1. `title`, `summary`, `body`를 정제하고 제목·요약이 없는 행을 보완합니다. 본문은 요약 보완에만 사용합니다.
2. 제목과 요약을 각각 임베딩합니다.
3. `제목 0.7 + 요약 0.3`으로 결합 벡터를 만들고 정규화합니다.
4. 카테고리마다 제목·요약·결합 벡터 `.npy` 3개와 행 매핑용 `.json` 1개를 저장합니다. CSV는 생성하지 않습니다.

In [ ]:
%pip install -q sentence-transformers pandas tqdm

In [13]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path('output')
EMBEDDING_DIR = Path('embeddings')
FILE_PATTERN = '*_enriched.csv'
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 64
MAX_SEQ_LENGTH = 512
SUMMARY_FALLBACK_CHARS = 500
TITLE_WEIGHT, SUMMARY_WEIGHT = 0.7, 0.3
TEXT_COLUMNS = ['title', 'summary', 'body']

EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split())

def fill_empty(primary: pd.Series, fallback: pd.Series) -> pd.Series:
    return primary.mask(primary.eq(''), fallback)

def preprocess_csv(path: Path) -> tuple[pd.DataFrame, dict]:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(TEXT_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name}: 필수 컬럼 누락 - {sorted(missing)}')

    for column in TEXT_COLUMNS:
        frame[column] = frame[column].map(clean_text)

    original_rows = len(frame)
    usable = frame[TEXT_COLUMNS].ne('').any(axis=1)
    dropped_source_rows = frame.index[~usable].tolist()
    frame = frame.loc[usable].copy()
    frame.insert(0, 'source_row', frame.index)
    frame.reset_index(drop=True, inplace=True)

    title_missing = frame['title'].eq('')
    summary_missing = frame['summary'].eq('')

    title_text = fill_empty(frame['title'], frame['summary'])
    title_text = fill_empty(title_text, frame['body'].str.slice(0, SUMMARY_FALLBACK_CHARS))

    summary_text = fill_empty(frame['summary'], frame['body'].str.slice(0, SUMMARY_FALLBACK_CHARS))
    summary_text = fill_empty(summary_text, title_text)

    frame['_embedding_title'] = title_text
    frame['_embedding_summary'] = summary_text
    frame['preprocess_note'] = [
        '; '.join(note) if note else 'original'
        for note in (
            (['title_filled'] if title else [])
            + (['summary_filled'] if summary else [])
            for title, summary in zip(title_missing, summary_missing)
        )
    ]

    report = {
        'file': path.name,
        'input_rows': original_rows,
        'usable_rows': len(frame),
        'dropped_all_empty': len(dropped_source_rows),
        'filled_title': int(title_missing.sum()),
        'filled_summary': int(summary_missing.sum()),
        'dropped_source_rows': dropped_source_rows,
    }
    return frame, report

In [15]:
csv_files = sorted(INPUT_DIR.glob(FILE_PATTERN))
if len(csv_files) != 6:
    raise RuntimeError(f'CSV 6개를 예상했지만 {len(csv_files)}개를 찾았습니다: {[p.name for p in csv_files]}')

prepared_data = {}
preprocess_reports = []
for path in csv_files:
    category = path.stem.removesuffix('_enriched')
    prepared_data[category], report = preprocess_csv(path)
    preprocess_reports.append(report)

preprocess_report_df = pd.DataFrame(preprocess_reports)
display(preprocess_report_df.drop(columns='dropped_source_rows'))
for report in preprocess_reports:
    if report['dropped_source_rows']:
        print(f"{report['file']} 제외 원본 행: {report['dropped_source_rows']}")

,file,input_rows,usable_rows,dropped_all_empty,filled_title,filled_summary
0,news_government_public_enriched.csv,144,144,0,0,0
1,news_industry_trends_enriched.csv,3373,3311,62,0,18
2,news_innodep_enriched.csv,39,39,0,0,0
3,news_production_wages_enriched.csv,31,31,0,0,0
4,news_security_enriched.csv,1011,1007,4,0,1
5,news_venture_finance_enriched.csv,1085,1065,20,0,7


news_industry_trends_enriched.csv 제외 원본 행: [36, 127, 142, 160, 230, 312, 318, 347, 369, 384, 444, 510, 517, 530, 679, 714, 740, 773, 781, 909, 962, 966, 996, 1018, 1024, 1119, 1153, 1154, 1183, 1218, 1276, 1330, 1364, 1488, 1533, 1540, 1542, 1553, 1575, 1586, 1607, 1645, 1655, 1664, 1754, 1809, 1877, 1891, 2025, 2054, 2059, 2070, 2157, 2262, 2421, 2480, 2615, 2641, 2691, 2709, 3269, 3370]
news_security_enriched.csv 제외 원본 행: [146, 197, 423, 565]
news_venture_finance_enriched.csv 제외 원본 행: [69, 72, 95, 127, 147, 267, 362, 503, 544, 597, 606, 610, 711, 726, 755, 762, 767, 800, 829, 898]


In [16]:
def normalize_rows(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.clip(norms, 1e-12, None)

def encode_field(model: SentenceTransformer, texts: pd.Series) -> np.ndarray:
    return model.encode(
        texts.tolist(),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32, copy=False)

def embed_category(category: str, frame: pd.DataFrame, model: SentenceTransformer) -> dict:
    print(f'[{category}] 제목 임베딩')
    title_vectors = encode_field(model, frame['_embedding_title'])
    print(f'[{category}] 요약 임베딩')
    summary_vectors = encode_field(model, frame['_embedding_summary'])

    combined_vectors = (
        TITLE_WEIGHT * title_vectors
        + SUMMARY_WEIGHT * summary_vectors
    )
    combined_vectors = normalize_rows(combined_vectors).astype(np.float32, copy=False)

    title_path = EMBEDDING_DIR / f'{category}_title_embeddings.npy'
    summary_path = EMBEDDING_DIR / f'{category}_summary_embeddings.npy'
    combined_path = EMBEDDING_DIR / f'{category}_embeddings.npy'
    metadata_path = EMBEDDING_DIR / f'{category}_metadata.json'
    np.save(title_path, title_vectors, allow_pickle=False)
    np.save(summary_path, summary_vectors, allow_pickle=False)
    np.save(combined_path, combined_vectors, allow_pickle=False)

    metadata_columns = [
        column for column in (
            'source_row', 'article_id', 'title', 'legacy_category',
            'article_link', 'source_file', 'fetch_status', 'preprocess_note',
        )
        if column in frame.columns
    ]
    metadata = {
        'model': MODEL_NAME,
        'dimension': int(combined_vectors.shape[1]),
        'weights': {'title': TITLE_WEIGHT, 'summary': SUMMARY_WEIGHT},
        'files': {
            'title': title_path.name,
            'summary': summary_path.name,
            'combined': combined_path.name,
        },
        'rows': frame[metadata_columns].to_dict(orient='records'),
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return {
        'category': category,
        'rows': len(combined_vectors),
        'dimension': combined_vectors.shape[1],
        'title_vectors': str(title_path),
        'summary_vectors': str(summary_path),
        'combined_vectors': str(combined_path),
        'metadata': str(metadata_path),
    }

In [17]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LENGTH
print(f'device={device} / model={MODEL_NAME} / max_seq_length={model.max_seq_length}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4222.57it/s]


device=cpu / model=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 / max_seq_length=512


## 카테고리별 임베딩

필요한 카테고리만 개별적으로 다시 실행할 수 있도록 셀을 분리했습니다.

In [18]:
government_public_result = embed_category(
    'news_government_public', prepared_data['news_government_public'], model
)
government_public_result

[news_government_public] 제목 임베딩


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.15it/s]


[news_government_public] 요약 임베딩


Batches: 100%|██████████| 3/3 [00:09<00:00,  3.18s/it]


{'category': 'news_government_public',
 'rows': 144,
 'dimension': 384,
 'title_vectors': 'embeddings\\news_government_public_title_embeddings.npy',
 'summary_vectors': 'embeddings\\news_government_public_summary_embeddings.npy',
 'combined_vectors': 'embeddings\\news_government_public_embeddings.npy',
 'metadata': 'embeddings\\news_government_public_metadata.json'}

In [19]:
industry_trends_result = embed_category(
    'news_industry_trends', prepared_data['news_industry_trends'], model
)
industry_trends_result

[news_industry_trends] 제목 임베딩


Batches: 100%|██████████| 52/52 [00:29<00:00,  1.76it/s]


[news_industry_trends] 요약 임베딩


Batches: 100%|██████████| 52/52 [01:52<00:00,  2.16s/it]


{'category': 'news_industry_trends',
 'rows': 3311,
 'dimension': 384,
 'title_vectors': 'embeddings\\news_industry_trends_title_embeddings.npy',
 'summary_vectors': 'embeddings\\news_industry_trends_summary_embeddings.npy',
 'combined_vectors': 'embeddings\\news_industry_trends_embeddings.npy',
 'metadata': 'embeddings\\news_industry_trends_metadata.json'}

In [20]:
innodep_result = embed_category(
    'news_innodep', prepared_data['news_innodep'], model
)
innodep_result

[news_innodep] 제목 임베딩


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


[news_innodep] 요약 임베딩


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]


{'category': 'news_innodep',
 'rows': 39,
 'dimension': 384,
 'title_vectors': 'embeddings\\news_innodep_title_embeddings.npy',
 'summary_vectors': 'embeddings\\news_innodep_summary_embeddings.npy',
 'combined_vectors': 'embeddings\\news_innodep_embeddings.npy',
 'metadata': 'embeddings\\news_innodep_metadata.json'}

In [21]:
production_wages_result = embed_category(
    'news_production_wages', prepared_data['news_production_wages'], model
)
production_wages_result

[news_production_wages] 제목 임베딩


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.72it/s]


[news_production_wages] 요약 임베딩


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.41s/it]


{'category': 'news_production_wages',
 'rows': 31,
 'dimension': 384,
 'title_vectors': 'embeddings\\news_production_wages_title_embeddings.npy',
 'summary_vectors': 'embeddings\\news_production_wages_summary_embeddings.npy',
 'combined_vectors': 'embeddings\\news_production_wages_embeddings.npy',
 'metadata': 'embeddings\\news_production_wages_metadata.json'}

In [22]:
security_result = embed_category(
    'news_security', prepared_data['news_security'], model
)
security_result

[news_security] 제목 임베딩


Batches: 100%|██████████| 16/16 [00:09<00:00,  1.67it/s]


[news_security] 요약 임베딩


Batches: 100%|██████████| 16/16 [00:30<00:00,  1.93s/it]


{'category': 'news_security',
 'rows': 1007,
 'dimension': 384,
 'title_vectors': 'embeddings\\news_security_title_embeddings.npy',
 'summary_vectors': 'embeddings\\news_security_summary_embeddings.npy',
 'combined_vectors': 'embeddings\\news_security_embeddings.npy',
 'metadata': 'embeddings\\news_security_metadata.json'}

In [23]:
venture_finance_result = embed_category(
    'news_venture_finance', prepared_data['news_venture_finance'], model
)
venture_finance_result

[news_venture_finance] 제목 임베딩


Batches: 100%|██████████| 17/17 [00:08<00:00,  1.99it/s]


[news_venture_finance] 요약 임베딩


Batches: 100%|██████████| 17/17 [00:40<00:00,  2.36s/it]


{'category': 'news_venture_finance',
 'rows': 1065,
 'dimension': 384,
 'title_vectors': 'embeddings\\news_venture_finance_title_embeddings.npy',
 'summary_vectors': 'embeddings\\news_venture_finance_summary_embeddings.npy',
 'combined_vectors': 'embeddings\\news_venture_finance_embeddings.npy',
 'metadata': 'embeddings\\news_venture_finance_metadata.json'}

In [24]:
result_df = pd.DataFrame([
    government_public_result, industry_trends_result, innodep_result,
    production_wages_result, security_result, venture_finance_result,
])
display(result_df)
print(f"완료: {len(result_df)}개 카테고리 / 총 {result_df['rows'].sum():,}개 벡터")

,category,rows,dimension,title_vectors,summary_vectors,combined_vectors,metadata
0,news_government_public,144,384,embeddings\news_government_public_title_embedd...,embeddings\news_government_public_summary_embe...,embeddings\news_government_public_embeddings.npy,embeddings\news_government_public_metadata.json
1,news_industry_trends,3311,384,embeddings\news_industry_trends_title_embeddin...,embeddings\news_industry_trends_summary_embedd...,embeddings\news_industry_trends_embeddings.npy,embeddings\news_industry_trends_metadata.json
2,news_innodep,39,384,embeddings\news_innodep_title_embeddings.npy,embeddings\news_innodep_summary_embeddings.npy,embeddings\news_innodep_embeddings.npy,embeddings\news_innodep_metadata.json
3,news_production_wages,31,384,embeddings\news_production_wages_title_embeddi...,embeddings\news_production_wages_summary_embed...,embeddings\news_production_wages_embeddings.npy,embeddings\news_production_wages_metadata.json
4,news_security,1007,384,embeddings\news_security_title_embeddings.npy,embeddings\news_security_summary_embeddings.npy,embeddings\news_security_embeddings.npy,embeddings\news_security_metadata.json
5,news_venture_finance,1065,384,embeddings\news_venture_finance_title_embeddin...,embeddings\news_venture_finance_summary_embedd...,embeddings\news_venture_finance_embeddings.npy,embeddings\news_venture_finance_metadata.json


완료: 6개 카테고리 / 총 5,597개 벡터
